In [57]:
import numpy as np

from scipy.stats import norm

def BS_call(X, t, K, T, R_grow, R_disc, sigma):
    """FINM 33000 L6 BS call formula.
    Returns time-t price of a T-expiry K-strike European call
    on a process X with growth rate R_grow and BS volatility sigma,
    discounting cashflows at R_disc.
    """
    tau = T - t
    F   = X * np.exp(R_grow * tau)
    sd  = sigma * np.sqrt(tau)
    d1  = np.log(F / K) / sd + 0.5 * sd
    d2  = d1 - sd
    return np.exp(-R_disc * tau) * (F * norm.cdf(d1) - K * norm.cdf(d2))

# Q1a

From

$$\mathbf{X}_T \;=\; \Big(\log S^{[j]}_0 + (r - \tfrac{1}{2}\sigma_{[j]}^2)T\Big)_{j=1,2} \;+\; \boldsymbol{\Sigma}\,\mathbf{W}_T,$$

where $\boldsymbol{\Sigma} = \text{diag}(\sigma_{[1]}, \sigma_{[2]})$ and $\mathbf{W}_T = (W^{[1]}_T, W^{[2]}_T)^\top$.

Then:

$$\text{Cov}(\mathbf{X}_T) \;=\; \mathbb{E}\big[\boldsymbol{\Sigma}\mathbf{W}_T(\boldsymbol{\Sigma}\mathbf{W}_T)^\top\big] \;=\; \boldsymbol{\Sigma}\,\mathbb{E}[\mathbf{W}_T \mathbf{W}_T^\top]\,\boldsymbol{\Sigma}^\top \;=\; T\,\boldsymbol{\Sigma}\,\text{Corr}(\mathbf{W}_T)\,\boldsymbol{\Sigma}^\top.$$

Since $\boldsymbol{\Sigma}$ is diagonal:

$$\boldsymbol{\Sigma}\,\text{Corr}\,\boldsymbol{\Sigma}^\top \;=\; \begin{pmatrix}\sigma_{[1]} & 0 \\ 0 & \sigma_{[2]}\end{pmatrix}\begin{pmatrix}1 & \rho \\ \rho & 1\end{pmatrix}\begin{pmatrix}\sigma_{[1]} & 0 \\ 0 & \sigma_{[2]}\end{pmatrix} \;=\; \begin{pmatrix}\sigma_{[1]}^2 & \rho\,\sigma_{[1]}\sigma_{[2]} \\ \rho\,\sigma_{[1]}\sigma_{[2]} & \sigma_{[2]}^2\end{pmatrix},$$

With $T = 1$, $\sigma_{[1]} = 0.3$, $\sigma_{[2]} = 0.2$, $\rho = 0.8$:

$$\;\text{Cov}(\mathbf{X}_T) \;=\; \begin{pmatrix}0.09 & 0.048 \\ 0.048 & 0.04\end{pmatrix}$$

## Problem 1

In [58]:
class MultiGBM:

    def __init__(self,S0,r,correlations,sigma):
        self.S0 = S0
        self.r = r
        self.correlations = correlations
        self.sigma = sigma

In [59]:
hw6p1dynamics = MultiGBM(S0=np.array([100,110]),r=0.05,
                         correlations = np.array([[1,0.8],[0.8,1]]),
                         sigma = np.diag([0.3, 0.2]))

In [60]:
class CallOnBasket:

    def __init__(self,K,T,weights):
        self.K = K
        self.T = T
        self.weights = weights

In [61]:
hw6p1contract=CallOnBasket(K=110,T=1.0,weights = np.array([1/2, 1/2]))

In [62]:
class MCengine:

    def __init__(self, M, antithetic, control, seed):
        self.M = M                                  # How many simulations
        self.antithetic = antithetic
        self.control = control
        self.rng = np.random.default_rng(seed=seed) # Seeding the random number generator with a specified number helps make the calculations reproducible

    def price_callonbasket_multiGBM(self,contract,dynamics): #CHANGED FROM GIVEN

        # You complete the coding of this function.
        # self.rng.multivariate_normal may be useful.
        # See documentation for numpy.random.Generator.multivariate_normal
        # as self.rng is an instance of numpy.random.Generator

        # You are not required to support the case where MC.control = MC.antithetic = True
        # (simultaneous use of control variate and antithetic)
        # But you are required to support the other 3 possible settings of MC.antithetic/MC.control
        # namely False/False, True/False, False/True.
        # (ordinary MC, antithetic without control, control without antithetic)


        S0    = dynamics.S0
        r     = dynamics.r
        Sigma = dynamics.sigma
        Corr  = dynamics.correlations
        T     = contract.T
        K     = contract.K
        w     = contract.weights
        M     = self.M
        d     = len(S0)

        Cov_XT = Sigma @ Corr @ Sigma.T * T
        sigma_vec = np.diag(Sigma)
        drift = (r - 0.5 * sigma_vec**2) * T

        # (False / False)
        if not self.antithetic and not self.control:

            Z = self.rng.multivariate_normal(mean=np.zeros(d), cov=Cov_XT, size=M)

            S_T  = S0 * np.exp(drift + Z)
            H_T  = S_T @ w
            Y    = np.exp(-r * T) * np.maximum(H_T - K, 0.0)

            call_price     = float(Y.mean())
            standard_error = float(Y.std(ddof=1) / np.sqrt(M))

            return (call_price, standard_error)
        
        # (True / False)
        if self.antithetic and not self.control:

            Z = self.rng.multivariate_normal(mean=np.zeros(d), cov=Cov_XT, size=M)

            S_T_plus  = S0 * np.exp(drift + Z)
            S_T_minus = S0 * np.exp(drift - Z)

            H_plus  = S_T_plus  @ w
            H_minus = S_T_minus @ w

            Y_plus  = np.exp(-r * T) * np.maximum(H_plus  - K, 0.0)
            Y_minus = np.exp(-r * T) * np.maximum(H_minus - K, 0.0)

            Y_av = 0.5 * (Y_plus + Y_minus)

            call_price     = float(Y_av.mean())
            standard_error = float(Y_av.std(ddof=1) / np.sqrt(M))

            return (call_price, standard_error)
        
        # (False / True)
        if not self.antithetic and self.control:

            Z = self.rng.multivariate_normal(mean=np.zeros(d), cov=Cov_XT, size=M)

            S_T = S0 * np.exp(drift + Z)
            H_T = S_T @ w
            G_T = np.sqrt(S_T[:, 0] * S_T[:, 1])

            disc  = np.exp(-r * T)
            Y     = disc * np.maximum(H_T - K, 0.0)
            Ystar = disc * np.maximum(G_T - K, 0.0)

            sigma_vec = np.diag(Sigma)
            s1, s2    = sigma_vec[0], sigma_vec[1]
            rho       = Corr[0, 1]
            sigma_G_sq = (s1**2 + 2*rho*s1*s2 + s2**2) / 4.0
            sigma_G    = np.sqrt(sigma_G_sq)
            R_grow_G   = r - (s1**2 + s2**2) / 4.0 + 0.5 * sigma_G_sq
            X_geom     = np.sqrt(S0[0] * S0[1])

            Cstar = BS_call(X=X_geom, t=0.0, K=K, T=T,R_grow=R_grow_G, R_disc=r, sigma=sigma_G)

            Y_bar     = Y.mean()
            Ystar_bar = Ystar.mean()
            cov_hat   = np.mean((Y - Y_bar) * (Ystar - Ystar_bar))
            var_hat   = np.mean((Ystar - Ystar_bar) ** 2)
            beta_hat  = cov_hat / var_hat

            Y_cv = Y + beta_hat * (Cstar - Ystar)

            call_price     = float(Y_cv.mean())
            standard_error = float(Y_cv.std(ddof=1) / np.sqrt(M))

            return (call_price, standard_error)


        return(call_price, standard_error)

In [63]:
hw6p1bMC=MCengine(M=10000,antithetic=False,control=False,seed=0)
(call_price_ordinary, std_err_ordinary) = hw6p1bMC.price_callonbasket_multiGBM(hw6p1contract,hw6p1dynamics)
print(call_price_ordinary, std_err_ordinary)

9.875007332598582 0.16838960569542308


In [64]:
hw6p1cMC=MCengine(M=10000,antithetic=True,control=False,seed=0)
(call_price_AV, std_err_AV) = hw6p1cMC.price_callonbasket_multiGBM(hw6p1contract,hw6p1dynamics)
print(call_price_AV, std_err_AV)

9.941127180624413 0.09518450200353225


# Q1d

Taking the log of the geometric mean gives a linear combination of the two log prices:

$$\log G_T \;=\; \tfrac{1}{2}\log S^{[1]}_T + \tfrac{1}{2}\log S^{[2]}_T \;=\; \tfrac{1}{2}\big(X^{[1]}_T + X^{[2]}_T\big)$$

where $X^{[j]}_T = \log S^{[j]}_T$. From the GBM SDE solution

$$X^{[j]}_T \;=\; \log S^{[j]}_0 \;+\; \Big(r - \tfrac{1}{2}\sigma_{[j]}^2\Big)T \;+\; \sigma_{[j]}\,W^{[j]}_T.$$

## Expectations

By linearity of expectation, and using $\mathbb{E}[W^{[j]}_T] = 0$:

$$\mathbb{E}[X^{[j]}_T] \;=\; \log S^{[j]}_0 + \Big(r - \tfrac{1}{2}\sigma_{[j]}^2\Big)T.$$

Therefore

$$
\begin{aligned}
\mathbb{E}[\log G_T] &= \tfrac{1}{2}\,\mathbb{E}[X^{[1]}_T] + \tfrac{1}{2}\,\mathbb{E}[X^{[2]}_T] \\
&= \tfrac{1}{2}\Big[\log S^{[1]}_0 + \log S^{[2]}_0\Big] + \tfrac{1}{2}\Big[(r - \tfrac{1}{2}\sigma_{[1]}^2) + (r - \tfrac{1}{2}\sigma_{[2]}^2)\Big]T \\
&= \tfrac{1}{2}\log(S^{[1]}_0 S^{[2]}_0) + \Big(r - \tfrac{\sigma_{[1]}^2 + \sigma_{[2]}^2}{4}\Big)T.
\end{aligned}
$$

## Variance

Only the Brownian part matters:

$$\log G_T - \mathbb{E}[\log G_T] \;=\; \tfrac{1}{2}\sigma_{[1]} W^{[1]}_T + \tfrac{1}{2}\sigma_{[2]} W^{[2]}_T.$$

Use $\text{Var}(aA + bB) = a^2\text{Var}(A) + 2ab\,\text{Cov}(A,B) + b^2\text{Var}(B)$ with $a = b = 1/2$:

$$
\begin{aligned}
\text{Var}(\log G_T) &= \tfrac{1}{4}\,\text{Var}(\sigma_{[1]} W^{[1]}_T) + 2\cdot\tfrac{1}{4}\,\text{Cov}(\sigma_{[1]} W^{[1]}_T,\,\sigma_{[2]} W^{[2]}_T) + \tfrac{1}{4}\,\text{Var}(\sigma_{[2]} W^{[2]}_T) \\
&= \tfrac{1}{4}\sigma_{[1]}^2 T + \tfrac{1}{2}\sigma_{[1]}\sigma_{[2]}\rho T + \tfrac{1}{4}\sigma_{[2]}^2 T \\
&= \tfrac{1}{4}\big(\sigma_{[1]}^2 + 2\rho\sigma_{[1]}\sigma_{[2]} + \sigma_{[2]}^2\big)T.
\end{aligned}
$$


# Q1e

From the function:

$$C^{BS}(X,\,t,\,K,\,T,\,R_{\text{grow}},\,R_{\text{disc}},\,\sigma)$$

The standard form looks like:

$$\log X_T \;\sim\; N\!\Big(\log X + \big(R_{\text{grow}} - \tfrac{1}{2}\sigma^2\big)(T-t),\;\sigma^2 (T-t)\Big)$$

Spot $X$ for the BS call: match $\log X$ to the log-spot piece of $\mathbb{E}[\log G_T]$:

$$\;X \;=\; \big(S^{[1]}_0\,S^{[2]}_0\big)^{1/2}\; $$
$$\log X = \tfrac{1}{2}\log(S^{[1]}_0 S^{[2]}_0)$$

Volatility $\sigma$: match $\sigma^2 T$ to $\text{Var}(\log G_T)$:

$$\;\sigma_G \;=\; \tfrac{1}{2}\sqrt{\sigma_{[1]}^2 + 2\rho\sigma_{[1]}\sigma_{[2]} + \sigma_{[2]}^2}\;$$

Discount rate $R_{\text{disc}}$: the actual cashflow at $T$ is paid in dollars, so we discount real cashflows at the actual risk-free rate $r$:

$$\;R_{\text{disc}} = r$$

Growth rate $R_{\text{grow}}$: match the drift part of the BS mean to our drift. The BS mean is $\log X + (R_{\text{grow}} - \tfrac{1}{2}\sigma_G^2)T$; ours is $\log X + (r - \tfrac{\sigma_{[1]}^2 + \sigma_{[2]}^2}{4})T$. Equating:

$$R_{\text{grow}} - \tfrac{1}{2}\sigma_G^2 \;=\; r - \tfrac{\sigma_{[1]}^2 + \sigma_{[2]}^2}{4}$$

$$\;R_{\text{grow}} \;=\; r - \tfrac{\sigma_{[1]}^2 + \sigma_{[2]}^2}{4} + \tfrac{1}{2}\sigma_G^2$$

So final:

Filling in the blanks from the PDF:

$$\;C^G \;=\; C^{BS}\!\Big(\,\big(S^{[1]}_0 S^{[2]}_0\big)^{1/2}\,,\; 0\,,\; K\,,\; T\,,\; \;R_{\text{grow}}\,,\; r\,,\; \sigma_G\,\Big)$$


In [65]:
hw6p1fMC=MCengine(M=10000,antithetic=False,control=True,seed=0)
(call_price_CV, std_err_CV) = hw6p1fMC.price_callonbasket_multiGBM(hw6p1contract,hw6p1dynamics)
print(call_price_CV, std_err_CV)

9.991036492142722 0.004401017044434682
